In [4]:
# notebooks/eda.py (run in a notebook for inline display; use `plt.savefig()` to save)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from pathlib import Path

sns.set(style="whitegrid")
FIG_DIR = Path("data/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# load processed CSV
df = pd.read_csv("data/phishing_email.csv")  
# columns in your dataset: ['text_combined','label'] or ['text','label']
text_col = "text_combined" if "text_combined" in df.columns else "text"
label_map = {0: "legitimate", 1: "phishing"}
df["label_name"] = df["label"].map(label_map)

# 1) Class distribution
plt.figure(figsize=(5,4))
sns.countplot(x="label_name", data=df)
plt.title("Class distribution")
plt.xlabel("")
plt.ylabel("Number of emails")
plt.tight_layout()
plt.savefig(FIG_DIR/"class_distribution.png", dpi=200)
plt.show()

# 2) Email length distribution (characters)
df["char_len"] = df[text_col].str.len()
plt.figure(figsize=(6,4))
sns.histplot(data=df, x="char_len", hue="label_name", bins=60, element="step", stat="count")
plt.xlim(0, 5000)  # trim extreme tail for clarity
plt.xlabel("Email length (chars)")
plt.title("Email length distribution by class")
plt.tight_layout()
plt.savefig(FIG_DIR/"length_distribution.png", dpi=200)
plt.show()

# 3) Top tokens per class (simple tokeniser)
def top_tokens(texts, n=20):
    words = Counter()
    for t in texts:
        toks = re.findall(r"\b[a-zA-Z]{2,}\b", str(t).lower())
        words.update([w for w in toks if len(w)>2 and w not in {"the","and","for","you","this"}])
    return words.most_common(n)

top_phish = top_tokens(df[df["label"]==1][text_col], n=20)
top_legit = top_tokens(df[df["label"]==0][text_col], n=20)

# bar chart side-by-side
ph_words, ph_counts = zip(*top_phish)
lg_words, lg_counts = zip(*top_legit)

fig, axes = plt.subplots(1,2,figsize=(12,5))
sns.barplot(x=list(ph_counts), y=list(ph_words), ax=axes[0])
axes[0].set_title("Top tokens — phishing")
sns.barplot(x=list(lg_counts), y=list(lg_words), ax=axes[1])
axes[1].set_title("Top tokens — legitimate")
plt.tight_layout()
plt.savefig(FIG_DIR/"top_tokens_comparison.png", dpi=200)
plt.show()

# 4) URL presence: crude detection of 'http' or 'www'
df["has_url"] = df[text_col].str.contains(r"https?://|www\.", regex=True, na=False)
url_tab = df.groupby("label_name")["has_url"].mean().reset_index()
url_tab["percent"] = url_tab["has_url"] * 100

plt.figure(figsize=(5,4))
sns.barplot(x="label_name", y="percent", data=url_tab)
plt.ylabel("% emails containing URL")
plt.title("URL presence by class")
plt.tight_layout()
plt.savefig(FIG_DIR/"url_presence.png", dpi=200)
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'data/phishing_email.csv'